In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist

from tensorflow.keras.utils import to_categorical

# Load and preprocess the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape((x_train.shape[0], 28, 28, 1)).astype('float32') / 255
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1)).astype('float32') / 255
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)


# Build the CNN model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy'])

# Train the model
model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.1)

# Evaluate the model
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")

In [ ]:
# TensorFlow Lite Konvertierung mit Quantisierung
rep_ds_size = 100
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.target_spec.supported_types = [tf.int8]

def representative_dataset_gen():
    """Representative dataset generator für Post-Training Quantisierung"""
    # Nutze nur die ersten rep_ds_size Samples
    for i in range(rep_ds_size):
        # Wichtig: Erweitere die Dimensionen um die Batch-Dimension
        sample = np.expand_dims(x_train[i], axis=0).astype(np.float32)
        yield [sample]

converter.representative_dataset = representative_dataset_gen
tflite_model = converter.convert()

# Speichere das quantisierte Modell
model_name  = "mnist_quantized_model"
model_path = model_name + '.tflite'
with open(model_path, 'wb') as f:
    f.write(tflite_model)

print(f"Quantisiertes Modell gespeichert als: {model_path}")


In [ ]:
# Optional: Teste das quantisierte Modell
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

# Hole Input- und Output-Details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Input shape: {input_details[0]['shape']}")
print(f"Input type: {input_details[0]['dtype']}")
print(f"Output shape: {output_details[0]['shape']}")
print(f"Output type: {output_details[0]['dtype']}")

# Teste mit einem Sample
test_sample = x_test[0:1].astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_sample)
interpreter.invoke()
tflite_result = interpreter.get_tensor(output_details[0]['index'])

print(f"Original prediction: {np.argmax(model.predict(test_sample))}")
print(f"TFLite prediction: {np.argmax(tflite_result)}")

Next, we simulate the OpenEye using `cocotb.test`. To do this, we have to create a `cocotb` test.

In [ ]:
import pytest
import cocotb_test
import cocotb_test.simulator

import os, sys

# change to pathlib
openeye_base = (os.path.abspath(os.path.join(os.pardir, os.pardir)))

tests_dir = os.path.abspath(os.path.join(openeye_base, "test"))
tb_dir = os.path.join(tests_dir, "cocotb_fpga")

hdl_dir = os.path.join(os.path.join(openeye_base, "hdl"))

sys.path.append(tests_dir)

print(tests_dir)

# TODO: Here is some creepy import not working properly
import test_utils.test_utils_main as tu

print(sys.path)


Define the parameters for the simulation:

In [ ]:
clk_cycle = 20
clk_cycle_unit = "ns"

clk_delay_in = 100
clk_delay_unit_in = "ps"

clk_delay_out = 100
clk_delay_unit_out = "ps"

In [ ]:
dut = 'OpenEye_FPGA'
module = 'OpenEye_FPGA_tb'
toplevel = dut

In [ ]:
verilog_sources = tu.get_verilog_sources(hdl_dir)

target_dir = os.path.join(tests_dir, 'simulation/' + model_name) 

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
import test_utils.open_eye_parameters as oe_params
import test_utils.vh_file_creator as vh_file_creator

In [ ]:
oep = oe_params.OpenEyeParameters()
vh_file_creator.create_vh_file(oep)

In [ ]:


results = cocotb_test.simulator.run(
    python_search=[tb_dir],
    verilog_sources=verilog_sources,
    toplevel=toplevel,
    module=module,
    sim_build=target_dir,
    testcase='model_test',
    force_compile=False,
    waves=True,
    includes=[hdl_dir],
    simulator="icarus",
    extra_env = {"CLOCK_LEN" : str(clk_cycle)
                ,"CLOCK_UNIT" : clk_cycle_unit
                ,"CLOCK_DELAY_INPUT" : str(clk_delay_in)
                ,"CLOCK_DELAY_UNIT_INPUT" : clk_delay_unit_in
                ,"CLOCK_DELAY_OUTPUT" : str(clk_delay_out)
                ,"CLOCK_DELAY_UNIT_OUTPUT" : clk_delay_unit_out
                ,"MODEL_PATH": model_path
                }
)